## STOCHASTIC MODELING
MODULE 6 | LESSON 4


---



# **RL for Asset Allocation: Q-learning & Deep Q-learning**

|  |  |
|:---|:---|
|**Reading Time** |  60 minutes |
|**Prior Knowledge** | MDP, Policy iteration, Portfolio rotation |
|**Keywords** | MDP, Q-learning, Deep Q-learning, epsilon-greedy, Neural Network

---


For the time period analyzed, our RL Portfolio rotation algorithm does a very decent job. However, the policy iteration approach developed required us to take a stand on transition probabilities ($P$) and rewards ($R$). We estimated this using history (during training), which seemed representative, but it can very well be the case that transition probabilities and rewards estimated in a given period or under a certain model do not serve as good estimates for out of sample use. To overcome this potential limitation, a branch of RL developed the so-called **Q-learning** approach, aiming at learning directly from experience about optimal actions.


In [ ]:
# Import libraries to be used later on
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


---

## 📚 Phần 0 — Lý thuyết nền tảng (Đọc trước khi code!)

---

### 🤖 Q-learning là gì?

**Q-learning** là một thuật toán học tăng cường (*Reinforcement Learning*) thuộc loại **model-free**: agent **không cần biết** xác suất chuyển trạng thái $P(s'|s,a)$ hay hàm phần thưởng $R(s,a)$ từ trước. Thay vào đó, agent **học trực tiếp từ kinh nghiệm** bằng cách thử-sai trong môi trường.

**Ý tưởng cốt lõi:** Học một hàm $Q(s, a)$ — giá trị kỳ vọng khi thực hiện hành động $a$ tại trạng thái $s$ và sau đó đi theo chính sách tối ưu.

#### Công thức cập nhật Q-table (Bellman Update):

$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha \Big[ \underbrace{r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a')}_{\text{TD target}} - Q(s_t, a_t) \Big]$$

Trong đó:
| Ký hiệu | Ý nghĩa |
|:--------|:--------|
| $\alpha$ | **Learning rate** — Agent học nhanh hay chậm (0 < α ≤ 1) |
| $\gamma$ | **Discount factor** — Agent coi trọng phần thưởng tương lai đến mức nào (0 ≤ γ ≤ 1) |
| $r_{t+1}$ | **Phần thưởng tức thời** nhận được sau khi thực hiện hành động $a_t$ |
| $\max_{a'} Q(s_{t+1}, a')$ | **Giá trị tốt nhất** agent có thể đạt ở bước tiếp theo |
| $Q(s_t, a_t)$ | **Ước lượng hiện tại** của giá trị hành động |

> 💡 **Hiểu đơn giản:** Agent liên tục "sửa" ước lượng của mình bằng cách so sánh những gì nó *nghĩ* sẽ xảy ra với những gì *thực sự* xảy ra. Đây chính là **Temporal Difference (TD) learning**.

---

### 🎮 Minh họa: Q-learning trong Gridworld

Hãy tưởng tượng một robot đang học cách đi trong một căn phòng ô vuông để tìm kho báu 🏆 và tránh hố 🕳️:

![Q-learning Gridworld Animation](https://miro.medium.com/v2/resize:fit:960/1*dL3zRL5p6QMOfbUm4W6Q0g.gif)

*GIF: Agent (robot) học dần cách đi từ ô xuất phát đến ô đích. Màu sắc thể hiện giá trị Q — càng đỏ càng tốt. Agent ban đầu đi ngẫu nhiên, rồi dần học được đường tối ưu.*

---

### 🎲 Chính sách ε-greedy (Exploration vs Exploitation)

Một vấn đề lớn trong RL: **nếu agent chỉ làm điều nó nghĩ là tốt nhất (exploitation), nó sẽ không bao giờ khám phá ra những hành động tốt hơn mà nó chưa thử.**

**ε-greedy** giải quyết điều này:

$$\pi(s) = \begin{cases} \text{hành động ngẫu nhiên} & \text{với xác suất } \varepsilon \text{ (Exploration)} \\ \arg\max_a Q(s, a) & \text{với xác suất } 1 - \varepsilon \text{ (Exploitation)} \end{cases}$$

![Exploration vs Exploitation](https://miro.medium.com/v2/resize:fit:640/format:webp/1*lBdYDKEEuiYj0dKFILiJsA.gif)

*GIF: Ban đầu ε cao → Agent khám phá nhiều (đi lăng quăng). Dần dần ε giảm → Agent khai thác kiến thức đã học.*

**Trong dự án này**, chúng ta dùng **epsilon decay**: ε giảm dần từ `eps_start` xuống `eps_end` qua các epoch để agent khám phá nhiều lúc đầu, rồi hội tụ về chính sách tối ưu.

---

### 🏦 Q-learning trong bài toán phân bổ tài sản

Trong bài toán **Portfolio Rotation** của chúng ta:

| Thành phần RL | Trong bài toán tài chính |
|:-------------|:------------------------|
| **State** $s$ | Trạng thái thị trường (12 trạng thái: momentum cổ phiếu × momentum trái phiếu × chế độ biến động) |
| **Action** $a$ | Tỷ lệ phân bổ (100% cổ phiếu / 75-25 / 50-50 / 25-75 / 100% trái phiếu) |
| **Reward** $r$ | Lợi nhuận danh mục kỳ tiếp theo |
| **Q-table** | Ma trận 12×5 lưu giá trị kỳ vọng của mỗi cặp (trạng thái, hành động) |

---


---

## 📚 Phần 0B — Deep Q-learning (DQN) là gì?

---

### 🧠 Giới hạn của Q-learning truyền thống

Q-learning truyền thống lưu toàn bộ giá trị $Q(s, a)$ trong một **bảng (Q-table)**. Điều này chỉ hoạt động tốt khi số trạng thái **nhỏ và rời rạc** (như 12 trạng thái trong bài của chúng ta).

**Vấn đề:** Nếu không gian trạng thái rất lớn (ví dụ: giá cổ phiếu liên tục, hình ảnh từ camera), bảng Q sẽ **quá lớn** để lưu trữ và học được.

### 🔗 Giải pháp: Thay Q-table bằng Neural Network

**Deep Q-Network (DQN)** thay thế bảng Q bằng một **mạng nơ-ron sâu** (neural network) có khả năng **xấp xỉ** hàm $Q(s, a)$:

$$Q(s, a; \theta) \approx Q^*(s, a)$$

Trong đó $\theta$ là các tham số (weights) của neural network.

![DQN Architecture](https://miro.medium.com/v2/resize:fit:720/format:webp/1*w5GuxedZ9ivRYqM_MLUxOQ.png)

*Hình: Kiến trúc DQN — Input là state, network tính toán Q-value cho tất cả actions, output là action có Q-value cao nhất.*

### 🎮 DQN chơi game Atari

DQN nổi tiếng nhờ DeepMind (2015) khi dạy AI chơi game Atari chỉ từ pixel hình ảnh:

![DQN Atari](https://miro.medium.com/v2/resize:fit:640/1*HlTMEn0dEI9C-IVeGnprog.gif)

*GIF: DQN học chơi game Breakout — Agent (thanh ngang) tự học chiến thuật "đào đường hầm" vào góc tường để ghi điểm cao nhất, mà không ai lập trình trước điều này!*

---

### ⚖️ So sánh Q-learning và Deep Q-learning

| Tiêu chí | Q-learning (cổ điển) | Deep Q-learning (DQN) |
|:---------|:---------------------|:----------------------|
| **Lưu trữ Q** | Bảng (Q-table) | Mạng nơ-ron (weights) |
| **Không gian trạng thái** | Nhỏ, rời rạc ✅ | Lớn, liên tục ✅ |
| **Khả năng tổng quát hóa** | Không (mỗi state học riêng) | Có (network xấp xỉ) |
| **Độ phức tạp** | Đơn giản 🟢 | Phức tạp hơn 🔴 |
| **Cần bộ nhớ lớn?** | Khi nhiều state | Ít hơn (nén vào weights) |
| **Ổn định khi huấn luyện** | Thường ổn định ✅ | Cần kỹ thuật: Experience Replay, Target Network |
| **Ứng dụng** | Gridworld nhỏ, bài toán tài chính đơn giản | Atari, robotics, xe tự lái |

### 🔑 Điểm giống nhau quan trọng:
- ✅ **Cả hai đều model-free**: không cần biết $P(s'|s,a)$ trước
- ✅ **Cùng mục tiêu**: tối ưu hóa Bellman equation
- ✅ **Cùng chiến lược**: ε-greedy để exploration/exploitation
- ✅ **Cùng công thức loss**: TD error = $r + \gamma \max Q(s') - Q(s,a)$

### 🔑 Điểm khác nhau cốt lõi:
- ❗ Q-learning cập nhật **trực tiếp vào bảng**; DQN cập nhật **weights của network** qua backpropagation
- ❗ DQN cần **Experience Replay** (lưu lại và lấy mẫu ngẫu nhiên từ quá khứ) để tránh overfitting
- ❗ DQN cần **Target Network** (một bản copy của network được cập nhật chậm hơn) để ổn định huấn luyện

---

> 🎯 **Trong project này, chúng ta sẽ cài đặt Q-learning cổ điển** (vì không gian trạng thái chỉ có 12 state). Nhưng hiểu DQN sẽ giúp em sẵn sàng cho các bài toán phức tạp hơn trong tương lai!

---


## **1. Q-learning**

The central change entailed by Q-learning is in the source of learning for our agent. Under the **policy iteration** approach, we estimated $R(s,a)$ and $P(s,s')$, then calculated value functions ($V^\pi$) and improved our policy until we achieved the optimal $\pi^*$.

**Q-learning** is a model-free approach: we do not need to estimate rewards nor transition probabilities, action values $Q(s,a)$ are directly learnt from the data.

**From state-value functions to action values ($Q$)**: In **Q-learning** the Bellman optimality is therefore set on $Q^*$:

$$
Q^*(s, a) = \mathbb{E}[r_{t+1} + \gamma \max_{a'} Q^*(s_{t+1}, a')| s_t = s, a_t = a]
$$

This Bellman equation contains both the **immediate reward** as well as the **best continuation value**. This implies:

1. No need for an explicit definition of $P(s, s')$

2. Policy evaluation and improvement collapse into one 'update'. Each time step $Q$ updates toward optimal.


So, the so-called **greedy policy** is just:

$$
\pi(s) = \arg\max_a Q(s,a)
$$


### 1.1. **Greedy and $ε$-greedy policies**

We have dealt already with greedy policies before implicitly, but not formally defined them. A **greedy policy** simply chooses the action with the highest known value. This implies some trade-offs:

- **Pros:**

  - A greedy policy is very simple to implement

  - Efficient in the long run when dealing with stable environments in which the optimal action is found quickly.

- **Cons:**

  - We face the risk of finding (and getting stuck in) a local optima.

  - Because of its potentially narrow focus, it does not let the agent explore other potentially higher rewards that require **exploration**.

**$\epsilon$-greedy** policies allow the agent to balance the trade-off between **exploration and exploitation**. In practice, an **$\epsilon$-greedy** policy will choose, with probability $ε$, an action at random (exploration); and with probability $1-\epsilon$ the action with the highest estimated value (exploitation).

- **Pros:**

  - Balance of the exploration-exploitation trade-off.

  - Avoids getting stuck in local optima.

- **Cons:**

  - Can be inefficient if $ε$ is too high.

  - Requires tuning the $\epsilon$ parameter.


### 1.2. How does the agent learn the optimal policy?

To answer this question, we need to understand how we are going to update $Q(s_t, a_t)$. For a transition $(s_t, a_t, r_{t+1}, s_{t+1})$, the **learning rule** is:

$$
Q(s_t, a_t) \leftarrow Q(s_t,a_t) + \alpha \left[r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t)  \right]
$$

where:

- $\alpha$ is the learning rate

- $\gamma$, as usual, is the discount rate

- The term $r_{t+1} + \gamma \max_{a'} Q(s_{t+1}, a')$ is usually called the **temporal difference (TD) target**.


## **2. Q-learning and Portfolio rotation**

To implement our first Q-learning algorithm, let's go back to the portfolio rotation example from Lesson 3. Let's start by loading the data:


In [ ]:
df = pd.read_csv("taa_monthly_data.csv", parse_dates=["date"], index_col="date")
df.head()


### 2.1. The portfolio rotation setup

As we know already, we need to define the RL setup. This will be equivalent to the one we defined in Lesson 3. Feel free to check it up for details on the code:


In [ ]:
# Rolling features
roll6_spy = (1 + df["ret_spy"]).rolling(6).apply(np.prod, raw=True) - 1
roll6_ief = (1 + df["ret_ief"]).rolling(6).apply(np.prod, raw=True) - 1
vol12_spy = df["ret_spy"].rolling(12).std() * np.sqrt(12)

df["eq_mom"]   = np.where(roll6_spy > 0, "pos", "neg")
df["bond_mom"] = np.where(roll6_ief > 0, "pos", "neg")
df["vol_raw"]  = vol12_spy

# Lag all three by 1 month to avoid look-ahead
df[["eq_mom","bond_mom","vol_raw"]] = df[["eq_mom","bond_mom","vol_raw"]].shift(1)

# Compute volatility tertiles on TRAIN (and apply same cutoffs to TEST sample to avoid leakage)
train_mask = (df.index >= "2004-01-31") & (df.index <= "2013-12-31")

q1, q2 = df.loc[train_mask, "vol_raw"].quantile([1/3, 2/3])

def vol_bucket(v):
    if pd.isna(v): return np.nan
    if v <= q1:    return "low"
    if v <= q2:    return "mid"
    return "high"

df["vol_regime"] = df["vol_raw"].map(vol_bucket)

# Map to state_id in 1 to 12 id
eq_map, bond_map, vol_map = {"neg":0,"pos":1}, {"neg":0,"pos":1}, {"low":0,"mid":1,"high":2}
idx_eq   = df["eq_mom"].map(eq_map)
idx_bond = df["bond_mom"].map(bond_map)
idx_vol  = df["vol_regime"].map(vol_map)
df["state_id"] = (idx_eq*6 + idx_bond*3 + idx_vol + 1).astype("Int64")

print(df[["eq_mom","bond_mom","vol_regime","state_id"]].dropna().tail(5))


We will also need to define actions (but no transition probs since we are in Q-learning!) and the usual **train**, **validation**, and **test** splits.


In [ ]:
ACTIONS = [(1.0,0.0),(0.75,0.25),(0.5,0.5),(0.25,0.75),(0.0,1.0)]  # Eq,Bond
S, A = 12, 5
gamma = 0.99

val_mask   = (df.index >= "2014-01-31") & (df.index <= "2017-12-31")
test_mask  = (df.index >= "2018-01-31")


---

### 2.2. ✏️ Bài tập 1: Cài đặt ε-greedy và TD update

**Nhiệm vụ của em:** Hoàn thành 2 hàm quan trọng nhất trong Q-learning bên dưới.

#### 📌 Gợi ý cho `select_action`:
- Tạo số ngẫu nhiên trong [0, 1). Nếu nhỏ hơn `epsilon` → chọn **ngẫu nhiên** (exploration)
- Ngược lại → chọn hành động có **Q-value cao nhất** trong hàng `s_idx` của bảng Q (exploitation)
- *Hint:* `np.random.rand()`, `np.random.randint(A)`, `np.flatnonzero(row == row.max())`

#### 📌 Gợi ý cho `td_update`:
- Tính **TD target** = phần thưởng tức thời + giá trị tối ưu kỳ tiếp (đã chiết khấu)
- Cập nhật $Q(s,a)$ bằng cách cộng thêm `alpha × (target - Q(s,a))`
- *Hint:* `Q[sp].max()` cho giá trị tối đa ở trạng thái tiếp theo

---


In [ ]:
def idx_state(s):   # map 1..12 -> 0..11
    return int(s) - 1

def select_action(Q, s_idx, epsilon):
    """
    ε-greedy policy:
    - Với xác suất epsilon: chọn hành động NGẪU NHIÊN (exploration)
    - Ngược lại: chọn hành động có Q-value CAO NHẤT (exploitation)
    
    Args:
        Q: Q-table có shape [S, A]
        s_idx: chỉ số trạng thái hiện tại (0-indexed)
        epsilon: xác suất khám phá
    Returns:
        chỉ số hành động được chọn
    """
    # === TODO: Điền code của em vào đây ===
    
    if ___________________:          # Nếu số ngẫu nhiên < epsilon → exploration
        return ___________________   # Trả về hành động ngẫu nhiên
    
    row = Q[s_idx]
    return np.random.choice(np.flatnonzero(row == ___________________))  # greedy w/ tie-break
    
    # =======================================

def td_update(Q, s, a, r, sp, alpha, gamma):
    """
    Cập nhật Q-table theo quy tắc Bellman:
    Q(s,a) ← Q(s,a) + alpha * [r + gamma * max_a' Q(s',a') - Q(s,a)]
    
    Args:
        Q: Q-table [S, A]
        s:  chỉ số trạng thái hiện tại
        a:  chỉ số hành động đã thực hiện
        r:  phần thưởng nhận được
        sp: chỉ số trạng thái tiếp theo (s')
        alpha: learning rate
        gamma: discount factor
    """
    # === TODO: Điền code của em vào đây ===
    
    target = r + gamma * ___________________   # TD target: r + γ * max Q(s', a')
    Q[s, a] += alpha * (___________________)   # Cập nhật Q(s, a)
    
    # =======================================

def r_next_equity_bond(row_next, a_idx):
    we, wb = ACTIONS[a_idx]
    return we*row_next.ret_spy + wb*row_next.ret_ief


---

### 2.3. ✏️ Bài tập 2: Cài đặt vòng lặp huấn luyện Q-learning

**Nhiệm vụ của em:** Hoàn thành hàm `train_qlearning` — đây là vòng lặp chính của thuật toán.

#### 📌 Thuật toán Q-learning (pseudocode):
```
Khởi tạo Q-table với giá trị optimistic
For mỗi epoch:
    Tính epsilon cho epoch này (linear decay)
    For mỗi bước thời gian t:
        s  = trạng thái tại t (convert sang 0-indexed)
        s' = trạng thái tại t+1
        a  = chọn hành động theo ε-greedy
        r  = tính phần thưởng (lợi nhuận danh mục)
        Cập nhật Q(s, a) theo TD update
Return Q
```

#### 📌 Gợi ý:
- **Epsilon decay**: `eps = eps_start + (eps_end - eps_start) * (e / max(1, epochs-1))`  
  → ε giảm **tuyến tính** từ `eps_start` xuống `eps_end` theo từng epoch
- Dùng các hàm đã cài ở Bài tập 1: `idx_state()`, `select_action()`, `td_update()`, `r_next_equity_bond()`

---


In [ ]:
def train_qlearning(df_train, alpha=0.10, eps_start=0.20, eps_end=0.01, epochs=150, optimistic=0.0):
    """
    Huấn luyện Q-table bằng thuật toán Q-learning.
    
    Args:
        df_train:   DataFrame dữ liệu training
        alpha:      Learning rate (tốc độ học)
        eps_start:  Epsilon ban đầu (nhiều exploration)
        eps_end:    Epsilon cuối (nhiều exploitation)
        epochs:     Số lần quét qua toàn bộ dữ liệu training
        optimistic: Giá trị khởi tạo ban đầu cho Q-table
    Returns:
        Q: Q-table đã được huấn luyện, shape [S, A]
    """
    # Bước 1: Khởi tạo Q-table với giá trị optimistic
    Q = np.full((S, A), ___________________, dtype=float)   # TODO: Dùng giá trị gì để khởi tạo?
    
    seq = df_train.dropna(subset=["state_id","ret_spy","ret_ief"])
    seq = seq[["state_id","ret_spy","ret_ief"]]

    for e in range(epochs):
        # Bước 2: Tính epsilon cho epoch này (linear decay từ eps_start xuống eps_end)
        eps = ___________________   # TODO: Công thức epsilon decay
        
        for t in range(len(seq)-1):
            # Bước 3: Lấy trạng thái hiện tại và tiếp theo
            s  = idx_state(___________________)   # TODO: Trạng thái tại bước t
            sp = idx_state(___________________)   # TODO: Trạng thái tại bước t+1
            
            # Bước 4: Chọn hành động theo ε-greedy
            a  = select_action(___________________)   # TODO: Gọi hàm select_action
            
            # Bước 5: Tính phần thưởng
            r  = r_next_equity_bond(___________________)   # TODO: Gọi hàm r_next_equity_bond
            
            # Bước 6: Cập nhật Q-table
            td_update(___________________)   # TODO: Gọi hàm td_update với đầy đủ tham số
    
    return Q


---

### 2.4. ✏️ Bài tập 3: Rút ra chính sách tối ưu từ Q-table

**Nhiệm vụ của em:** Hoàn thành hàm `greedy_policy_from_Q` — sau khi học xong, agent sẽ luôn chọn hành động tốt nhất.

#### 📌 Gợi ý:
- Với mỗi trạng thái (hàng của Q-table), chọn cột có **giá trị Q cao nhất**
- `Q.argmax(axis=1)` cho chỉ số cột max trên mỗi hàng
- Kết quả là một dictionary: `{state_id (1..12): best_action (1..5)}`

---


In [ ]:
ACTIONS_DICT = {1:(1.0,0.0), 2:(0.75,0.25), 3:(0.5,0.5), 4:(0.25,0.75), 5:(0.0,1.0)}

def greedy_policy_from_Q(Q):
    """
    Rút ra chính sách tham lam từ Q-table đã huấn luyện.
    Với mỗi trạng thái, chọn hành động có Q-value cao nhất.
    
    Args:
        Q: Q-table đã huấn luyện, shape [S, A]
    Returns:
        dict: {state_id: best_action_id} với state 1..12, action 1..5
    """
    # === TODO: Điền code của em vào đây ===
    
    best = Q.argmax(___________________)   # TODO: argmax theo chiều nào? (axis=?)
    return {s: int(a+1) for s, a in zip(___________________, best)}   # TODO: state từ 1 đến S+1
    
    # =======================================

def backtest_policy(df_slice, policy_dict, cost_bps=10):
    """Backtest hiệu quả của policy, bao gồm chi phí giao dịch (transaction costs)."""
    out = df_slice.copy()
    out = out.dropna(subset=["state_id","ret_spy","ret_ief"])
    out["action"] = out["state_id"].map(policy_dict).astype("Int64")

    we = out["action"].map(lambda a: ACTIONS_DICT[int(a)][0] if pd.notna(a) else np.nan)
    r_e_next = out["ret_spy"].shift(-1)
    r_b_next = out["ret_ief"].shift(-1)
    r_gross  = we * r_e_next + (1 - we) * r_b_next

    we_prev  = we.shift(1)
    turnover = (we - we_prev).abs().fillna(0.0)
    cost = (cost_bps / 10_000.0) * turnover
    r_net = (r_gross - cost).dropna()

    curve = (1 + r_net).cumprod()
    return r_net, curve


---

### 2.5. ✏️ Bài tập 4: Tinh chỉnh hyperparameters (Grid Search)

**Nhiệm vụ của em:** Mở rộng lưới tìm kiếm `grid` bằng cách thêm ít nhất **2 bộ hyperparameters mới** mà em cho là có thể cho kết quả tốt hơn.

#### 📌 Gợi ý để chọn hyperparameters:
- **`alpha` nhỏ hơn** (0.01–0.05): học chậm hơn nhưng ổn định hơn
- **`epochs` nhiều hơn** (300–500): cho agent thêm thời gian học
- **`eps_start` cao hơn** (0.4–0.5): khám phá nhiều hơn ban đầu
- **`optimistic` khác nhau** (0.001, 0.01): thử mức "lạc quan" khác nhau

Câu hỏi suy nghĩ: *Tại sao chúng ta không dùng test set để chọn hyperparameters?*

---


#### Gợi ý trả lời:

> *(Viết câu trả lời của em vào đây)*
>
> Chúng ta không dùng test set để chọn hyperparameters vì...


In [ ]:
grid = [
    # --- Baselines (no optimism) ---
    {"alpha":0.10, "epochs":150, "eps_start":0.20, "eps_end":0.01, "optimistic":0.00},
    {"alpha":0.05, "epochs":200, "eps_start":0.20, "eps_end":0.01, "optimistic":0.00},
    {"alpha":0.08, "epochs":250, "eps_start":0.15, "eps_end":0.01, "optimistic":0.00},
    {"alpha":0.12, "epochs":150, "eps_start":0.25, "eps_end":0.02, "optimistic":0.00},

    # --- With mild optimism ---
    {"alpha":0.10, "epochs":150, "eps_start":0.15, "eps_end":0.01, "optimistic":0.002},
    {"alpha":0.05, "epochs":250, "eps_start":0.20, "eps_end":0.02, "optimistic":0.002},
    {"alpha":0.08, "epochs":200, "eps_start":0.20, "eps_end":0.01, "optimistic":0.002},
    {"alpha":0.12, "epochs":200, "eps_start":0.25, "eps_end":0.01, "optimistic":0.002},

    # --- Stronger optimism ---
    {"alpha":0.10, "epochs":200, "eps_start":0.15, "eps_end":0.01, "optimistic":0.005},
    {"alpha":0.05, "epochs":300, "eps_start":0.20, "eps_end":0.01, "optimistic":0.005},
    {"alpha":0.08, "epochs":250, "eps_start":0.25, "eps_end":0.02, "optimistic":0.005},
    {"alpha":0.12, "epochs":200, "eps_start":0.20, "eps_end":0.01, "optimistic":0.005},

    # --- Lower ε floor ---
    {"alpha":0.10, "epochs":250, "eps_start":0.20, "eps_end":0.005, "optimistic":0.00},
    {"alpha":0.08, "epochs":300, "eps_start":0.15, "eps_end":0.005, "optimistic":0.002},

    # --- Higher ε start ---
    {"alpha":0.10, "epochs":150, "eps_start":0.30, "eps_end":0.02, "optimistic":0.00},
    {"alpha":0.05, "epochs":250, "eps_start":0.30, "eps_end":0.01, "optimistic":0.002},

    # =====================================================
    # === TODO: Thêm ít nhất 2 bộ hyperparameters mới ===
    # =====================================================
    # {"alpha":???, "epochs":???, "eps_start":???, "eps_end":???, "optimistic":???},
    # {"alpha":???, "epochs":???, "eps_start":???, "eps_end":???, "optimistic":???},
]


In [ ]:
def perf_from_monthly(r):
    r = r.dropna(); n = len(r); yrs = n/12
    cagr = (1+r).prod()**(1/yrs) - 1
    vol  = r.std() * np.sqrt(12)
    sr   = cagr/vol if vol>0 else np.nan
    return cagr, vol, sr

df_train = df.loc[train_mask]
df_val   = df.loc[val_mask]
results = []

print(f"Starting grid search with {len(grid)} configurations...\n")

for i, hp in enumerate(grid, 1):
    print(f"[{i}/{len(grid)}] Testing: alpha={hp['alpha']}, epochs={hp['epochs']}, "
          f"eps_start={hp['eps_start']}, eps_end={hp['eps_end']}, optimistic={hp['optimistic']}")
    
    Q = train_qlearning(df_train, **hp)
    pi = greedy_policy_from_Q(Q)
    r_val, _ = backtest_policy(df_val, pi, cost_bps=10)
    cagr, vol, sr = perf_from_monthly(r_val)
    
    print(f"    → Ann Return: {cagr:6.2%}, Ann Vol: {vol:6.2%}, Sharpe: {sr:5.2f}\n")
    
    results.append((hp, cagr, vol, sr))

print("Grid search complete!")

val_tbl = pd.DataFrame([
    {
        "alpha":hp["alpha"], "epochs":hp["epochs"], "eps_start":hp["eps_start"], "eps_end":hp["eps_end"],
        "optimistic":hp["optimistic"], "Ann Return":c, "Ann Vol":v, "Sharpe":s
    } for hp,c,v,s in results
])
val_tbl.sort_values("Sharpe", ascending=False, inplace=True)
val_tbl


In [ ]:
# Pick best set of parameters in validation based on Sharpe Ratio
best_hp = val_tbl.iloc[0][["alpha","epochs","eps_start","eps_end","optimistic"]].to_dict()
best_hp["epochs"] = int(best_hp["epochs"])
Q_best  = train_qlearning(df_train, **best_hp)
PI_q    = greedy_policy_from_Q(Q_best)

# Test period
df_test = df.loc[test_mask]
r_q, curve_q = backtest_policy(df_test, PI_q, cost_bps=10)

# 60/40 baseline performance
r_6040 = (0.6*df_test["ret_spy"].shift(-1) + 0.4*df_test["ret_ief"].shift(-1)).dropna()
curve_6040 = (1 + r_6040).cumprod()

# Performance table (CAGR, Vol, Sharpe)
def make_perf_table(r_q, r_b):
    def row(r):
        c,v,s = perf_from_monthly(r); return [c,v,s]
    tbl = pd.DataFrame(
        [row(r_q), row(r_b)],
        index=["Q-learning policy","60/40"],
        columns=["Annualized Return","Annualized Volatility","Sharpe (rf=0)"]
    )
    pretty = tbl.copy()
    pretty["Annualized Return"]     = pretty["Annualized Return"].map(lambda x: f"{x:.2%}")
    pretty["Annualized Volatility"] = pretty["Annualized Volatility"].map(lambda x: f"{x:.2%}")
    pretty["Sharpe (rf=0)"]         = pretty["Sharpe (rf=0)"].map(lambda x: f"{x:.2f}")
    return tbl, pretty

perf_raw, perf_pretty = make_perf_table(r_q, r_6040)
print(perf_pretty)


It seems that the performance of the two strategies now is very close!


---

### ✏️ Bài tập 5: Vẽ biểu đồ kết quả và phân tích

**Nhiệm vụ của em:** Hoàn thành đoạn code vẽ biểu đồ **hiệu suất tích lũy** của Q-learning vs 60/40, sau đó trả lời câu hỏi phân tích phía dưới.

#### 📌 Gợi ý:
- `curves` là DataFrame chứa 2 cột: `"Q-learning"` và `"60/40"`
- Dùng `plt.plot()` để vẽ từng đường, thêm `label=` để phân biệt
- Nhớ thêm `plt.title()`, `plt.xlabel()`, `plt.ylabel()`, `plt.legend()`

---


In [ ]:
import matplotlib.pyplot as plt

curves = pd.DataFrame({"Q-learning": curve_q, "60/40": curve_6040}).dropna()

plt.figure(figsize=(10, 5))

# === TODO: Vẽ đường hiệu suất cho Q-learning ===
plt.plot(___________________, ___________________, label=___________________)

# === TODO: Vẽ đường hiệu suất cho 60/40 ===
plt.plot(___________________, ___________________, label=___________________)

# === TODO: Thêm tiêu đề và nhãn trục ===
plt.title(___________________)
plt.xlabel(___________________)
plt.ylabel(___________________)

plt.legend()
plt.tight_layout()
plt.show()


---

### ✏️ Bài tập 6 (Suy nghĩ): Phân tích kết quả

Nhìn vào kết quả, hãy trả lời các câu hỏi sau. Viết câu trả lời của em vào ô bên dưới mỗi câu:

---

**Câu 1:** Q-learning hoạt động tốt hơn hay kém hơn chiến lược 60/40 trên tập test? Dựa vào chỉ số nào để em kết luận?

> *(Viết câu trả lời của em vào đây)*

---

**Câu 2:** Kể ra ít nhất **2 nguyên nhân** có thể giải thích tại sao Q-learning không vượt trội hoàn toàn so với Policy Iteration trong bài toán này:

> *(Viết câu trả lời của em vào đây)*

---

**Câu 3 (nâng cao):** Nếu em được phép dùng **Deep Q-learning (DQN)** thay cho Q-learning cổ điển trong bài này, em nghĩ điều đó sẽ giúp ích hay không? Tại sao? Liệu không gian trạng thái 12 state có đủ lớn để cần DQN không?

> *(Viết câu trả lời của em vào đây)*

---

**Câu 4 (nâng cao):** Epsilon decay giúp gì cho quá trình học? Điều gì sẽ xảy ra nếu chúng ta giữ epsilon **cố định ở 0.5** trong suốt quá trình training?

> *(Viết câu trả lời của em vào đây)*

---


## **3. Conclusion**

Well done! In this lesson we have implemented a Portfolio rotation algorithm using Q-learning. As you can see, the performance for this specific task of policy iteration and Q-learning is slightly different. Keep in mind we have not tuned the algorithm to its fullest, so there's some potential room for improvement.

### 📌 Tóm tắt những gì em đã học:

| Kiến thức | Nội dung |
|:---------|:---------|
| **Q-learning** | Model-free RL: học trực tiếp từ kinh nghiệm, không cần biết P và R |
| **ε-greedy** | Cân bằng exploration (khám phá) vs exploitation (khai thác) |
| **TD Update** | Cập nhật Q(s,a) dần dần qua từng bước thời gian |
| **Epsilon decay** | ε giảm dần để agent khai thác nhiều hơn khi đã học đủ |
| **Hyperparameter tuning** | Dùng validation set (không phải test set!) để chọn params |
| **Deep Q-learning** | Thay Q-table bằng neural network cho bài toán phức tạp hơn |

We will dig deeper into how to use more complex computational methods for portfolio management and trading, among other things, in subsequent courses. In the next module we will introduce another important topic: *Network theory*

---


---
Copyright 2025 WorldQuant University. This
content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.
